In [5]:
# --- 라이브러리 임포트 ---
import os                                  # ← 폴더 내부 파일을 탐색하기 위해 os 모듈 불러와
import re                                  # ← 정규표현식 처리를 위해 re 모듈 불러와
import unicodedata                         # ← 유니코드 정규화(NFKC) 위해 불러와
import csv                                 # ← CSV 파싱 옵션(quoting 상수) 위해 불러와
import pandas as pd                        # ← 엑셀, csv 파일을 다루기 위해 pandas 불러와

# --- 유틸: 셀/컬럼명 표준화 (탭/개행/특수공백 제거 + 다중 공백 1칸) ---
def _norm_text(s):                         # ← 임의의 텍스트 s를 표준화하는 함수 정의
    s = unicodedata.normalize('NFKC', str(s))   # ← 전각/반각 등 형태를 통일(NFKC)
    s = s.replace('\u00A0', ' ')           # ← non-breaking space를 일반 공백으로
    s = re.sub(r'[\t\r\n]+', ' ', s)       # ← 탭/개행을 공백으로 치환
    s = re.sub(r'\s+', ' ', s)             # ← 연속 공백을 단일 공백으로
    return s.strip()                        # ← 좌우 공백을 제거해서 반환

# --- 유틸: 컬럼명 표준화 (전체 컬럼에 _norm_text 적용) ---
def normalize_columns(df):                  # ← 데이터프레임 df의 컬럼명을 표준화하는 함수
    df.columns = [_norm_text(c) for c in df.columns]  # ← 모든 컬럼명에 표준화 적용
    return df                               # ← 표준화된 df 반환

# --- 유틸: 완전 빈 컬럼(전부 공백/NaN) 제거 ---
def drop_empty_columns(df):                 # ← 의미 없는 빈 컬럼 제거 함수
    to_drop = []                            # ← 제거 대상 컬럼명을 담을 리스트
    for c in df.columns:                    # ← 모든 컬럼 순회
        col = df[c]                         # ← 해당 컬럼 시리즈
        if col.isna().all():                # ← 전부 NaN이면
            to_drop.append(c)               # ← 제거 대상에 추가
        else:                               # ← NaN만은 아니면
            as_str = col.astype(str).str.strip()   # ← 문자열 변환 후 좌우 공백 제거
            if (as_str == '').all():        # ← 전부 빈 문자열이면
                to_drop.append(c)           # ← 제거 대상에 추가
    if to_drop:                             # ← 제거 대상이 존재하면
        df = df.drop(columns=to_drop)       # ← 해당 컬럼 제거
    return df                               # ← 정리된 df 반환

# --- 유틸: 'Unnamed:'로 시작하는 잡열 제거 ---
def drop_unnamed_columns(df):               # ← Unnamed 계열 컬럼 제거 함수
    return df.loc[:, ~df.columns.str.startswith('Unnamed:')]  # ← 이름이 'Unnamed:'로 시작하는 컬럼 제외

# --- '일자'/'품목코드' 계열 패턴(문자 사이 공백 허용) ---
DATE_PATTERNS = [r'일\s*자', r'전\s*표\s*일\s*자', r'등\s*록\s*일\s*자', r'기\s*준\s*일\s*자', r'날\s*짜']  # ← 일자 후보
CODE_PATTERNS = [r'품\s*목\s*코\s*드', r'상\s*품\s*코\s*드', r'아\s*이\s*템\s*코\s*드', r'품\s*번']      # ← 품목코드 후보

# --- 유틸: 패턴 리스트 중 하나라도 포함되는지 검사 ---
def _matches_any(text, patterns):           # ← text에 patterns 중 하나라도 매칭되는지 확인
    return any(re.search(p, text) for p in patterns)  # ← 정규식 매칭 결과 True/False 반환

# --- 유틸: 한 행을 하나의 문자열로 합쳐 비교 (헤더 탐지용) ---
def row_to_text(row_values):                # ← 행의 값들을 표준화 후 하나의 문자열로 합침
    return ' | '.join([_norm_text(v) for v in row_values])  # ← 표준화 + 구분자 삽입하여 합치기

# --- XLSX: 정규식 기반 헤더 행 탐지 ---
def find_header_row_xlsx(sheet_df):         # ← header=None으로 읽은 전체 시트에서 헤더 위치를 탐지
    for i, row in sheet_df.iterrows():      # ← 모든 행을 순회
        row_text = row_to_text(row.values)  # ← 행 전체를 표준화 텍스트로 변환
        if _matches_any(row_text, DATE_PATTERNS) and _matches_any(row_text, CODE_PATTERNS):  # ← 같은 행에 일자/품목코드가 함께 있으면
            return i                         # ← 해당 행 인덱스를 헤더로 반환
    return None                              # ← 못 찾으면 None 반환

# --- 읽은 뒤: '일자'/'품목코드' 컬럼명 통일 ---
def unify_core_headers(df):                 # ← df 컬럼명을 패턴 매칭으로 통일
    new_cols = []                           # ← 통일된 컬럼명을 담을 리스트
    for c in df.columns:                    # ← 원래 컬럼명 순회
        s = _norm_text(c)                   # ← 컬럼명 표준화
        if re.search('|'.join(DATE_PATTERNS), s):   # ← 일자 계열이면
            new_cols.append('일자')          # ← '일자'로 통일
        elif re.search('|'.join(CODE_PATTERNS), s): # ← 품목코드 계열이면
            new_cols.append('품목코드')       # ← '품목코드'로 통일
        else:                               # ← 그 외는
            new_cols.append(s)              # ← 표준화된 원래 이름 사용
    df.columns = new_cols                   # ← 컬럼명 교체
    return df                               # ← df 반환

# --- CSV: 견고한 읽기 (인코딩 재시도 + 따옴표 밖 콤마만 분리 + 문자열 우선) ---
def robust_read_csv(path):                  # ← CSV를 안정적으로 읽는 함수
    last_error = None                       # ← 마지막 오류 저장 변수
    for enc in ('utf-8-sig', 'cp949'):      # ← 흔한 인코딩 후보 순서
        try:
            df = pd.read_csv(               # ← CSV 전체를 읽기
                path,                       # ← 파일 경로
                sep=r',(?=(?:[^"]*"[^"]*")*[^"]*$)',  # ← 따옴표 밖 콤마만 분리(정규식)
                engine='python',            # ← 정규식 sep 쓰려면 python 엔진
                dtype=str,                  # ← 전부 문자열로(타입 혼선 방지)
                keep_default_na=False,      # ← 'NA' 같은 문자열을 NaN 처리하지 않도록
                quoting=csv.QUOTE_MINIMAL,  # ← 기본 따옴표 처리
                on_bad_lines='skip',        # ← 불량행은 건너뛰기
                encoding=enc                # ← 인코딩 후보 적용
            )
            return df                       # ← 성공 시 df 반환
        except Exception as e:              # ← 실패하면
            last_error = e                  # ← 마지막 오류 갱신
            continue                        # ← 다음 인코딩으로 재시도
    raise last_error                        # ← 모두 실패 시 오류 발생

# --- 입력 폴더/출력 파일 설정 ---
root_dir = "C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4"  # ← 원본 파일들이 있는 루트 폴더
output_file = '251014_본사재고수불부모음_ver5.xlsx'         # ← 결과 저장 파일명

# --- 합쳐진 데이터를 저장할 빈 데이터프레임 생성 ---
combined_df = pd.DataFrame()               # ← 최종 병합 그릇 생성

# --- 폴더 내부를 전부 탐색 (하위 폴더까지) ---
for dirpath, dirnames, filenames in os.walk(root_dir):  # ← 폴더 재귀 탐색 시작
    for filename in filenames:                           # ← 파일들 하나씩 확인
        if (filename.lower().endswith(('.xlsx', '.csv'))) and not filename.startswith('~$'):  # ← 엑셀/CSV만, 임시(~$) 제외
            file_path = os.path.join(dirpath, filename)  # ← 파일 전체 경로 생성

            try:                                         # ← 파일별 예외 처리
                # -----------------------
                # 엑셀 파일(.xlsx) 처리
                # -----------------------
                if filename.lower().endswith('.xlsx'):   # ← 엑셀 분기
                    xls = pd.ExcelFile(file_path)        # ← 엑셀 핸들 열기(시트 목록 확인)
                    for sheet_name in xls.sheet_names:   # ← 모든 시트 순회
                        # 1) 헤더 없이 전체 읽어서 헤더 행을 정규식 기반으로 탐지
                        sheet_df = xls.parse(sheet_name, header=None)  # ← 탐지용 전체 읽기
                        header_row_index = find_header_row_xlsx(sheet_df)  # ← 헤더 행 탐지

                        # 2) 그래도 못 찾으면 상단 20행 묶음 텍스트로 보조 탐지
                        if header_row_index is None:      # ← 1차 탐지 실패 시
                            top = sheet_df.head(20)       # ← 상단 20행만
                            flat = row_to_text(top.values.ravel())     # ← 평탄화 텍스트
                            if _matches_any(flat, DATE_PATTERNS) and _matches_any(flat, CODE_PATTERNS):  # ← 상단 블럭에 두 키워드가 있다면
                                found = None             # ← 가장 아래쪽 후보를 찾기 위한 변수
                                for i, row in top.iloc[::-1].iterrows():  # ← 위에서 아래로가 아닌, 아래에서 위로
                                    t = row_to_text(row.values)  # ← 행 텍스트
                                    if _matches_any(t, DATE_PATTERNS) and _matches_any(t, CODE_PATTERNS):  # ← 두 키워드 함께?
                                        found = i       # ← 해당 행 인덱스 기록
                                        break           # ← 종료
                                header_row_index = found  # ← 보조 탐지 결과 반영

                        # 3) 헤더 없으면 스킵
                        if header_row_index is None:      # ← 끝까지 못 찾으면
                            print(f"⚠️ 헤더 미탐지로 스킵: {file_path} - {sheet_name}")  # ← 경고 로그
                            continue                      # ← 다음 시트로 넘어가

                        # 4) 실제 데이터 읽기 (헤더 줄까지 스킵 → 바로 아래가 컬럼행)
                        df = pd.read_excel(               # ← 실제 데이터 부분 읽기
                            file_path,                    # ← 파일 경로
                            sheet_name=sheet_name,        # ← 시트명
                            skiprows=header_row_index     # ← 헤더 줄까지 건너뛰기
                        )

                        # 5) 전처리: Unnamed 제거 → 빈 컬럼 제거 → 컬럼명 표준화 → 핵심 컬럼 통일
                        df = drop_unnamed_columns(df)     # ← 'Unnamed:'로 시작하는 컬럼 제거
                        df = drop_empty_columns(df)       # ← 완전 빈 컬럼 제거
                        df = normalize_columns(df)        # ← 컬럼명 탭/개행/특수공백 정리
                        df = unify_core_headers(df)       # ← '일자'/'품목코드' 등 표준 컬럼명으로 통일

                        # 6) 파일/시트 메타 정보 추가
                        df['파일명'] = filename           # ← 파일명 메모
                        df['파일_경로'] = file_path       # ← 파일 경로 메모
                        df['시트명'] = sheet_name         # ← 시트명 메모
                        df['상위_폴더'] = os.path.basename(dirpath)  # ← 상위 폴더명 메모

                        # 7) 최종 병합
                        combined_df = pd.concat([combined_df, df], ignore_index=True)  # ← 누적 병합
                        print(f"✅ {file_path} - {sheet_name} (헤더 {header_row_index}행) 데이터 합침 완료!")  # ← 진행 로그

                # -----------------------
                # CSV 파일(.csv) 처리
                # -----------------------
                elif filename.lower().endswith('.csv'):   # ← CSV 분기
                    # 1) 견고하게 읽기 (인코딩 재시도 + 따옴표 밖 콤마만 분리 + 문자열 우선)
                    df = robust_read_csv(file_path)       # ← CSV 안정 로딩

                    # 2) 전처리: Unnamed 제거 → 빈 컬럼 제거 → 컬럼명 표준화 → 핵심 컬럼 통일
                    df = drop_unnamed_columns(df)         # ← 'Unnamed:' 제거
                    df = drop_empty_columns(df)           # ← 완전 빈 컬럼 제거
                    df = normalize_columns(df)            # ← 컬럼명 표준화
                    df = unify_core_headers(df)           # ← '일자'/'품목코드' 통일

                    # 3) 파일 메타 정보 추가
                    df['파일명'] = filename               # ← 파일명 메모
                    df['파일_경로'] = file_path           # ← 파일 경로 메모
                    df['시트명'] = 'CSV'                  # ← CSV는 시트 개념 없으니 고정
                    df['상위_폴더'] = os.path.basename(dirpath)  # ← 상위 폴더명 메모

                    # 4) 최종 병합
                    combined_df = pd.concat([combined_df, df], ignore_index=True)  # ← 누적 병합
                    print(f"✅ {file_path} - CSV 데이터 정상 합침 완료!")  # ← 진행 로그

            except Exception as e:                        # ← 파일 단위 예외
                print(f"❌ {file_path} 파일 오류: {e}")    # ← 오류 로그 출력

# --- 결과 저장 (엑셀) ---
combined_df = normalize_columns(combined_df)            # ← 혹시 모를 이중 공백/탭 제거 재확인
combined_df = drop_unnamed_columns(combined_df)         # ← 혹시 섞여 들어온 Unnamed 재제거
combined_df.to_excel(output_file, index=False, engine='openpyxl')  # ← 엑셀로 저장
print(f"\n🎉 수정된 전체 데이터를 '{output_file}'로 저장 완료!")  # ← 저장 완료 로그


✅ C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4\01.xlsx - 01 (헤더 1행) 데이터 합침 완료!
✅ C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4\02.xlsx - 02 (헤더 1행) 데이터 합침 완료!
✅ C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4\03.xlsx - 03 (헤더 1행) 데이터 합침 완료!
✅ C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4\04.xlsx - 04 (헤더 1행) 데이터 합침 완료!


C:\Users\jusun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\jusun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4\05.xlsx - 재고수불부 (헤더 1행) 데이터 합침 완료!


C:\Users\jusun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\jusun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4\06.xlsx - 재고수불부 (헤더 1행) 데이터 합침 완료!
✅ C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4\07.xlsx - 07 (헤더 1행) 데이터 합침 완료!


...

✅ C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4\58.xlsx - 재고수불부 (헤더 1행) 데이터 합침 완료!


C:\Users\jusun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\jusun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4\59.xlsx - 재고수불부 (헤더 1행) 데이터 합침 완료!


C:\Users\jusun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\jusun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4\60.xlsx - 재고수불부 (헤더 1행) 데이터 합침 완료!
❌ C:/Users/jusun/OneDrive/문서/3yejoo/ERP/재고/251002_재고수불부/1004본사재고수불부_ver4\합친파일.xlsx 파일 오류: Excel file format cannot be determined, you must specify an engine manually.

🎉 수정된 전체 데이터를 '251014_본사재고수불부모음_ver5.xlsx'로 저장 완료!
